In [0]:
import pandas as pd
# Note: keep_default_na=False prevents pandas from turning "NA" (North America) into NaN
sales_df = pd.read_csv(
    "/Volumes/workspace/sales/sales/sales_transactions.csv",
    keep_default_na=False,
    na_values=[""],
    parse_dates=["posting_date"],
)

In [0]:
# Quick look at what we're dealing with
print("Shape:", sales_df.shape)
print(sales_df.dtypes)
print("\nMissing values:")
print(sales_df.isna().sum())
print("\nDuplicates:", sales_df.duplicated().sum())
print("\nSample of region values:", sales_df["region"].unique())
print("Sample of material codes:", sales_df["material_code"].unique())

Shape: (5839, 8)
document_no              object
posting_date     datetime64[ns]
customer_id               int64
material_code            object
plant_code               object
quantity_mt             float64
net_value_eur            object
region                   object
dtype: object

Missing values:
document_no      0
posting_date     0
customer_id      0
material_code    0
plant_code       0
quantity_mt      2
net_value_eur    3
region           0
dtype: int64

Duplicates: 3

Sample of region values: ['EMEA' 'LATAM' 'NA' 'APAC' ' emea ' 'apac' 'Latam' 'NA ' 'EMEA ']
Sample of material codes: ['MAT-RD-02' 'MAT-RD-03' 'MAT-SAF-01' 'MAT-RN-01' 'MAT-RD-01' 'MAT-RN-02'
 'MAT-SAF-02' 'mat-rd-01' 'MAT-SAF-01 ' ' mat-rn-01']


In [0]:
sales_df

,document_no,posting_date,customer_id,material_code,plant_code,quantity_mt,net_value_eur,region
0,SO-2023-000000,2023-04-13,100034,MAT-RD-02,PLANT-FI,169.96,216331.07,EMEA
1,SO-2023-000001,2024-03-11,100005,MAT-RD-03,PLANT-FI,258.32,398386.98,EMEA
2,SO-2023-000002,2023-09-28,100027,MAT-SAF-01,PLANT-US,297.36,590136.21,LATAM
3,SO-2023-000003,2023-04-17,100023,MAT-RD-02,PLANT-FI,537.77,680937.23,EMEA
4,SO-2023-000004,2023-03-13,100040,MAT-RN-01,PLANT-FI,617.75,563307.68,EMEA
...,...,...,...,...,...,...,...,...
5834,SO-2024-005834,2024-02-05,100033,MAT-RD-01,PLANT-NL,541.14,770115.08,EMEA
5835,SO-2024-005835,2024-11-11,100012,MAT-RD-02,PLANT-US,191.96,219303.56,NA
5836,SO-2023-000050,2024-01-23,100020,MAT-RD-02,PLANT-SG,544.55,693662.58,APAC
5837,SO-2023-001000,2023-11-25,100007,MAT-SAF-01,PLANT-NL,438.74,881764.51,EMEA


In [0]:
# --- Clean region
sales_df["region"] = sales_df["region"].str.strip().str.upper()

In [0]:
sales_df["region"].unique()

array(['EMEA', 'LATAM', 'NA', 'APAC', ' emea ', 'apac', 'Latam', 'NA ',
       'EMEA '], dtype=object)

In [0]:
# --- Clean material_code
sales_df["material_code"] = sales_df["material_code"].str.strip().str.upper()

In [0]:
sales_df["material_code"].unique()

array(['MAT-RD-02', 'MAT-RD-03', 'MAT-SAF-01', 'MAT-RN-01', 'MAT-RD-01',
       'MAT-RN-02', 'MAT-SAF-02'], dtype=object)

In [0]:
# --- Clean net_value_eur
sales_df["net_value_eur"] = pd.to_numeric(
    sales_df["net_value_eur"]
        .astype(str)
        .str.replace("€", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
    errors="coerce",
)

In [0]:
sales_df["net_value_eur"].head()

0    216331.07
1    398386.98
2    590136.21
3    680937.23
4    563307.68
Name: net_value_eur, dtype: object

In [0]:
# --- Clean quantity_mt
sales_df["quantity_mt"] = pd.to_numeric(sales_df["quantity_mt"], errors="coerce")

In [0]:
sales_df["quantity_mt"].dtype

dtype('float64')

In [0]:
sales_df["quantity_mt"].describe()

count     5837.000000
mean       442.811828
std       1321.198750
min         50.190000
25%        237.930000
50%        427.450000
75%        614.890000
max      99999.000000
Name: quantity_mt, dtype: float64

In [0]:
# --- Flag outliers in quantity_mt (anything above 20,000 MT is suspicious)
sales_df.loc[sales_df["quantity_mt"] > 20000, "quantity_mt"] = pd.NA

In [0]:
sales_df["quantity_mt"].max()

np.float64(799.94)

In [0]:
# --- Drop duplicate rows
before = len(sales_df)
sales_df = sales_df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(sales_df)} duplicate row(s)")

Removed 3 duplicate row(s)


In [0]:
sales_df.duplicated().sum()

np.int64(0)

In [0]:
# --- Final overview ---
sales_df.shape

(5836, 8)

In [0]:
sales_df.dtypes

document_no              object
posting_date     datetime64[ns]
customer_id               int64
material_code            object
plant_code               object
quantity_mt             float64
net_value_eur           float64
region                   object
dtype: object

In [0]:
sales_df.isna().sum()

document_no      0
posting_date     0
customer_id      0
material_code    0
plant_code       0
quantity_mt      3
net_value_eur    3
region           0
dtype: int64

In [0]:
(spark.createDataFrame(sales_df)
    .write
    .mode("overwrite")
    .saveAsTable("workspace.sales.sales_transactions_2024"))

In [0]:
%sql
Select *
From workspace.sales.sales_transactions_2024
Limit 10

document_no,posting_date,customer_id,material_code,plant_code,quantity_mt,net_value_eur,region
SO-2023-001945,2023-09-27T00:00:00.000Z,100003,MAT-SAF-01,PLANT-FI,696.51,1294162.69,EMEA
SO-2023-001946,2024-08-14T00:00:00.000Z,100011,MAT-RN-02,PLANT-US,368.25,395691.93,NA
SO-2023-001947,2024-08-27T00:00:00.000Z,100024,MAT-SAF-02,PLANT-SG,736.77,1551995.01,APAC
SO-2023-001948,2023-12-27T00:00:00.000Z,100022,MAT-RD-02,PLANT-NL,423.45,523377.93,EMEA
SO-2023-001949,2023-09-14T00:00:00.000Z,100026,MAT-SAF-02,PLANT-US,349.13,646561.21,LATAM
SO-2023-001950,2023-05-13T00:00:00.000Z,100013,MAT-RN-01,PLANT-FI,612.79,698987.68,EMEA
SO-2023-001951,2023-10-03T00:00:00.000Z,100001,MAT-RN-01,PLANT-FI,104.88,132263.15,EMEA
SO-2023-001952,2024-11-13T00:00:00.000Z,100024,MAT-RN-02,PLANT-SG,117.11,110918.84,APAC
SO-2023-001953,2024-05-27T00:00:00.000Z,100010,MAT-RD-03,PLANT-FI,538.98,815215.67,EMEA
SO-2023-001954,2023-11-09T00:00:00.000Z,100035,MAT-SAF-02,PLANT-NL,415.21,779115.14,EMEA
